# Proof of Concept — Notebook A (Instructor + Team Red)

Starts the competition (instructor) and plays as `team_red`. Run Notebook B
(Team Blue) in a separate kernel pointing at the SAME `SERVER_IP`/`SERVER_PORT`.

In [8]:
from pynqsim import SimulationClient

SERVER_IP = "172.20.166.171"   # <-- your server IP
SERVER_PORT = 9003             # <-- your server port

red = SimulationClient(SERVER_IP, SERVER_PORT)
print("Team Red connected to server!")

Team Red connected to server!


## Instructor: start the competition
If you get `Competition already active`, stop it first with
`red.admin_stop_competition(password=ADMIN_PASSWORD)`.

In [9]:
ADMIN_PASSWORD = "admin123"

# Per-team JOIN passwords. Each team must supply its own to join.
JOIN_PASSWORDS = {
    "team_red":  "red123",
    "team_blue": "blue123",
}

# ---------------------------------------------------------------------
# Author the board EXACTLY as it looks top-down (5 rows x 6 cols).
# board_to_grid() rotates it into the array orientation the scene
# expects (6x5) -- the SCENE FILE IS UNCHANGED. This transform is
# verified to reproduce the previous working CARD_LAYOUT cell-for-cell.
# ---------------------------------------------------------------------
BOARD_VIEW = [
    ["dog",     "aeroplane",    "person",    "bottle",       "cat",    "boat"],           # top row
    ["bird",    "car",          "cow",       "dining table", "sheep",  "bicycle"],
    ["chair",   "potted plant", "boat",      "dog",          "sofa",   "person"],
    ["cow",     "potted plant", "aeroplane", "bird",         "bottle", "dining table"],
    ["bicycle", "cat",          "chair",     "car",          "sofa",   "sheep"],           # bottom row
]

def board_to_grid(board):
    """Rotate a top-down board view (BR x BC) into the scene's array
    orientation (BC x BR).  grid[r][c] = board[BR-1-c][BC-1-r]."""
    BR, BC = len(board), len(board[0])
    assert all(len(row) == BC for row in board), "BOARD_VIEW rows must be equal length"
    GR, GC = BC, BR
    return [[board[BR - 1 - c][BC - 1 - r] for c in range(GC)] for r in range(GR)]

CARD_LAYOUT = {"grid": board_to_grid(BOARD_VIEW)}

# Sanity check: each object should appear exactly twice.
from collections import Counter
_counts = Counter(name for row in BOARD_VIEW for name in row)
_bad = {k: v for k, v in _counts.items() if v != 2}
print(f"{sum(_counts.values())} cells, {len(_counts)} unique labels")
if _bad:
    print("WARNING: not used exactly twice:", _bad)

red.admin_start_competition(
    "competition_card_flip",
    password=ADMIN_PASSWORD,
    card_layout=CARD_LAYOUT,
    join_passwords=JOIN_PASSWORDS,
)
print("Competition started! Team Blue can now join from the other notebook.")

30 cells, 15 unique labels
Competition started! Team Blue can now join from the other notebook.


In [10]:
# Team Red joins with its join password (set by the instructor above).
red.join_competition(team_id="team_red", password="red123")
print("Joined as team_red!")

Joined as team_red!


In [11]:
state = red.get_competition_state()
print(f"Current turn: {state.get('current_turn')}")
print(f"Scores: {state.get('scores', {})}")

Current turn: team_red
Scores: {'team_red': 0, 'team_blue': 0}


## Color names (6x5 grid → 15 colors)
These MUST match the order the scene assigns `color_idx` in
`competition_card_flip.py` (indices 0..14).

In [12]:
# 15 colors, in the exact order the scene assigns color_idx (0..14).
COLOR_NAMES = [
    "RED", "GRN", "BLU", "YEL", "ORG",
    "PUR", "BRN", "PNK", "CYN", "MAG",
    "LIM", "TEA", "GRY", "MAR", "NVY",
]

def color_name(ci):
    return COLOR_NAMES[ci] if (ci is not None and ci < len(COLOR_NAMES)) else str(ci)

## Normal play (`flip`)
Drives the RED arm with full memory-game scoring: a **match** lets the same team
go again; a **mismatch** re-covers the cards and passes the turn.

In [ ]:
def flip(client, row, col):
    """Flip a card WITH memory-game scoring/turn messaging."""
    result = client.flip_card(row, col)
    if result.get("already_flipped"):
        print(f"Card ({row},{col}) already flipped")
        return result
    print(f">>> ({row},{col}) revealed: {color_name(result['color_idx'])}")
    mr = result.get("match_result")
    if mr:
        if mr.get("matched"):
            print("*** MATCH! +1 — same team goes again! ***")
        elif mr.get("turn_ended"):
            print("No match. Cards re-covered. Turn passes to the other team.")
    return result


flip(red, 5, 2)  # first card
flip(red, 5, 1)  # second card

## `flip_raw()` — grab a cover with NO scoring / turn / re-cover
This calls the **`flip_raw` server action**. Unlike `flip_card`, it does **not**
run `record_flip`, so there's no scorekeeping, no turn check, and mismatched
covers are **not** teleported back — the cover just goes to the discard pile and
stays there. That's what lets a single client sweep the whole board in one loop.

`flip_raw` takes an explicit `robot_id` because one arm can't reach all 30 cards —
use **red (0)** for the left columns and **blue (1)** for the right columns.

In [15]:
def flip_raw(client, row, col, robot_id=None):
    """Grab one cover: no scoring, no turn logic, no re-cover."""
    params = {"row": row, "col": col}
    if robot_id is not None:
        params["robot_id"] = robot_id
    result = client._request("flip_raw", params)
    if result.get("already_flipped"):
        print(f"({row},{col}) already flipped")
    else:
        print(f"grabbed ({row},{col}) -> {color_name(result.get('color_idx'))}")
    return result


flip_raw(red, 2, 0)
flip_raw(red, 2, 1)

grabbed (2,0) -> CYN
grabbed (2,1) -> MAG


{'color_idx': 9, 'already_flipped': False, 'matched': False, 'status': 'ok'}

## `unflip_raw()` — admin force re-cover of a single card
Teleports the cover back onto a card. This is an **admin** action (needs the
admin password), so only the instructor can command it. It force-covers ANY
card regardless of flipped/matched state, with no scoring or turn logic.
Uses ARRAY (row, col) coordinates, same as `flip_raw`.
Requires the server `admin_cover_card` action + `simulation.cover_card` method.

In [16]:
def unflip_raw(row, col):
    """ADMIN: teleport the cover back onto (row, col). Password-gated."""
    result = red._request("admin_cover_card", {
        "password": ADMIN_PASSWORD,
        "row": row,
        "col": col,
    })
    print(f"re-covered ({row},{col}): {result.get('covered')}")
    return result


# Example: put the covers back on the two cards flipped above.
unflip_raw(2, 0)
unflip_raw(2, 1)

re-covered (2,0): True
re-covered (2,1): True


{'status': 'ok', 'covered': True}

## Reset the board (instructor)
Re-covers every card and zeroes both teams' scores, keeping the SAME layout.
Use for a clean board mid-session instead of restarting the competition.
Requires the `admin_reset_board` server action + client method.

In [ ]:
def reset_board():
    """Instructor: put all covers back on and zero the scores."""
    red.admin_reset_board(password=ADMIN_PASSWORD)
    state = red.get_competition_state()
    print("Board reset.")
    print(f"Scores: {state.get('scores', {})}")
    print(f"Current turn: {state.get('current_turn')}")

reset_board()

## Cleanup

In [ ]:
red.leave_competition()
print("Team Red left.")
# red.admin_stop_competition(password=ADMIN_PASSWORD)